# Retrieval v2 + BGE + grounded QA — Colab end-to-end

Notebook này tạo **đúng submission public cũ** (`100 query = 50 TKIS + 50 VKIS`, mỗi query 100 answer), nhưng bắt buộc chạy BGE-M3 dense retrieval và BGE cross-encoder rerank cho TKIS. Sau khi submission hoàn tất, notebook chạy smoke riêng cho KIS, AVS và QA; chỉ QA load Qwen3.5 grounded answerer.

Không có ASR. QA smoke không được trộn vào `submission.csv` vì public contract hiện tại không có task QA.


## Cách chạy và nguyên tắc resume

1. Chọn GPU runtime, mount Drive và chỉnh cell `parameters`.
2. Chạy tuần tự từ trên xuống. Lần đầu cần mạng để tải model.
3. Nếu runtime ngắt, giữ nguyên `RUN_ID`; đặt `START_AT` bằng stage đầu tiên chưa `passed` trong `run_manifest.json`.
4. `BGE_*_MODE = "required"` bảo đảm model mới lỗi thì run fail, không âm thầm trả submission theo kiến trúc cũ.
5. Qwen chạy ở process riêng sau pipeline submission để GPU của process trước đã được giải phóng.


In [ ]:
# parameters — chỉ sửa giá trị trong cell này
from datetime import datetime, timezone
from pathlib import Path

SOURCE_MODE = "drive"  # "git" hoặc "drive"
GIT_URL = "https://github.com/24122013/AIChallenge26_Multimodal_Agentic_Video_Retrieval_System.git"
GIT_BRANCH = "feature/new_qa_new_model"
DRIVE_REPO_PATH = "/content/drive/MyDrive/AIChallenge26/repository"

PUBLIC_DATA_SOURCE = "/content/drive/MyDrive/AIChallenge26/public"
DRIVE_RUNS_ROOT = "/content/drive/MyDrive/AIChallenge26/retrieval_runs"
MODEL_CACHE_ROOT = "/content/drive/MyDrive/AIChallenge26/model_cache"
RUN_ID = datetime.now(timezone.utc).strftime("new-model-%Y%m%d-%H%M%S")

PADDLE_GPU_PROFILE = "cu126"  # "cu118" hoặc "cu126"
DEVICE = "cuda"
BATCH_SIZE = "auto"
NUM_WORKERS = 2
CANDIDATE_INTERVAL_SEC = 0.5
MAX_GAP_SECONDS = 2.0
START_AT = "validate-input"
STOP_AFTER = "validate-submission"

BGE_DENSE_MODE = "required"
BGE_RERANKER_MODE = "required"
BGE_M3_MODEL = "BAAI/bge-m3"
BGE_M3_REVISION = "main"  # pin commit hash trước benchmark chính thức
BGE_RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"
BGE_RERANKER_REVISION = "main"  # pin commit hash trước benchmark chính thức
BGE_RERANKER_ALPHA = 0.5
BGE_BATCH_SIZE = 8

RUN_TASK_SMOKE = True
TASK_SMOKE = "all"  # kis | avs | qa | all
TASK_TOP_K = 5
KIS_QUERY = "người phụ nữ mặc áo đỏ đang cầm điện thoại"
AVS_QUERY = "tất cả các cảnh có xe máy đi qua đường"
QA_QUERY = "Người phụ nữ mặc áo đỏ đang cầm vật gì?"
QA_ANSWER_MODE = "required"
QA_MODEL = "Qwen/Qwen3.5-9B"
QA_MODEL_REVISION = "c202236"
QA_QUANTIZATION = "4bit"

WORKSPACE = Path("/content/ai_challenge26")
PUBLIC_ROOT = Path("/content/public_data")
RUN_ROOT = Path(DRIVE_RUNS_ROOT) / RUN_ID
SUBMISSION_PATH = RUN_ROOT / "results" / "submission.csv"
TASK_SMOKE_PATH = RUN_ROOT / "results" / "task_smoke.json"


## 1. GPU, Drive và source code

In [ ]:
import os
import shutil
import subprocess
import sys

from google.colab import drive

drive.mount("/content/drive")
gpu = subprocess.run(["nvidia-smi"], text=True, capture_output=True, check=False)
print(gpu.stdout[-5000:] if gpu.stdout else gpu.stderr)
if gpu.returncode != 0:
    raise RuntimeError("Không thấy NVIDIA GPU. Hãy đổi Colab runtime sang GPU.")

if SOURCE_MODE == "git":
    if WORKSPACE.exists():
        shutil.rmtree(WORKSPACE)
    subprocess.run(
        ["git", "clone", "--branch", GIT_BRANCH, "--single-branch", GIT_URL, str(WORKSPACE)],
        check=True,
    )
elif SOURCE_MODE == "drive":
    source = Path(DRIVE_REPO_PATH)
    if not (source / "competition" / "run_retrieval_v2.py").is_file():
        raise FileNotFoundError(f"Drive repo không hợp lệ: {source}")
    if WORKSPACE.exists():
        shutil.rmtree(WORKSPACE)
    shutil.copytree(
        source,
        WORKSPACE,
        ignore=shutil.ignore_patterns(".git", ".venv", "__pycache__", "data", "artifacts"),
    )
else:
    raise ValueError("SOURCE_MODE phải là 'git' hoặc 'drive'")

os.chdir(WORKSPACE)
print("workspace:", WORKSPACE)
print("branch:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip() if (WORKSPACE / ".git").exists() else "drive-copy")


## 2. Cài dependency GPU và kiểm tra runtime

Notebook cài một requirements chung và đúng **một** Paddle GPU profile. Không cài `requirements.txt` vì file đó là profile CPU.


In [ ]:
profile = WORKSPACE / f"requirements-gpu-{PADDLE_GPU_PROFILE}.txt"
if not profile.is_file():
    raise FileNotFoundError(f"GPU profile không tồn tại: {profile}")

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-core.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(profile)], check=True)

import paddle
import torch
import transformers

runtime = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "torch_cuda": torch.cuda.is_available(),
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "paddle": paddle.__version__,
    "paddle_cuda": paddle.is_compiled_with_cuda(),
    "transformers": transformers.__version__,
}
print(runtime)
if DEVICE == "cuda" and (not runtime["torch_cuda"] or not runtime["paddle_cuda"]):
    raise RuntimeError("Torch/Paddle GPU runtime chưa sẵn sàng; kiểm tra lại PADDLE_GPU_PROFILE và restart runtime.")


## 3. Chuẩn bị public dataset và khóa contract 100 query

In [ ]:
source = Path(PUBLIC_DATA_SOURCE)
required = ("corpus.csv", "questions.csv", "sample_submission.csv")
missing = [name for name in required if not (source / name).is_file()]
if missing:
    raise FileNotFoundError(f"PUBLIC_DATA_SOURCE thiếu: {missing}")

if PUBLIC_ROOT.exists():
    shutil.rmtree(PUBLIC_ROOT)
shutil.copytree(source, PUBLIC_ROOT)
RUN_ROOT.mkdir(parents=True, exist_ok=True)

subprocess.run(
    [sys.executable, "-m", "competition.pipeline", "validate-input", "--public-root", str(PUBLIC_ROOT)],
    check=True,
)


## 4. Chạy end-to-end và tạo submission cũ bằng model mới

Pipeline có 10 stage. `bge-text-index` build embedding từ metadata đã có; `predict` dùng BGE-M3 + BM25/visual fusion và BGE reranker cho TKIS. VKIS vẫn search bằng ảnh. Legacy SmolVLM rerank để `off` vì không phải Qwen QA.


In [ ]:
command = [
    sys.executable, "-m", "competition.run_retrieval_v2",
    "--public-root", str(PUBLIC_ROOT),
    "--run-root", str(RUN_ROOT),
    "--submission-path", str(SUBMISSION_PATH),
    "--device", DEVICE,
    "--batch-size", BATCH_SIZE,
    "--num-workers", str(NUM_WORKERS),
    "--candidate-interval-sec", str(CANDIDATE_INTERVAL_SEC),
    "--max-gap-seconds", str(MAX_GAP_SECONDS),
    "--model-cache-root", str(MODEL_CACHE_ROOT),
    "--start-at", START_AT,
    "--stop-after", STOP_AFTER,
    "--vlm-mode", "off",
    "--bge-dense-mode", BGE_DENSE_MODE,
    "--bge-reranker-mode", BGE_RERANKER_MODE,
    "--bge-m3-model-name", BGE_M3_MODEL,
    "--bge-m3-model-revision", BGE_M3_REVISION,
    "--bge-reranker-model-name", BGE_RERANKER_MODEL,
    "--bge-reranker-model-revision", BGE_RERANKER_REVISION,
    "--bge-reranker-alpha", str(BGE_RERANKER_ALPHA),
    "--bge-batch-size", str(BGE_BATCH_SIZE),
]
print(" ".join(map(str, command)))
subprocess.run(command, cwd=WORKSPACE, check=True)


## 5. Xác nhận submission và bằng chứng model mới đã chạy

In [ ]:
import csv
import json

manifest_path = RUN_ROOT / "run_manifest.json"
trace_path = RUN_ROOT / "results" / "query_traces.jsonl"
bge_manifest_path = RUN_ROOT / "indexes" / "bge_m3" / "bge_m3_manifest.json"

for path in (SUBMISSION_PATH, manifest_path, trace_path, bge_manifest_path):
    if not path.is_file():
        raise FileNotFoundError(path)

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
failed = [name for name, state in manifest.get("stages", {}).items() if state.get("status") != "passed"]
if failed:
    raise RuntimeError(f"Stage chưa passed: {failed}")
if len(manifest.get("stages", {})) != 10:
    raise RuntimeError("run_manifest phải có đúng 10 stage của kiến trúc mới")

with SUBMISSION_PATH.open(encoding="utf-8-sig", newline="") as stream:
    rows = list(csv.reader(stream))
if len(rows) != 101 or len(rows[0]) != 101:
    raise RuntimeError(f"submission sai shape: rows={len(rows)-1}, columns={len(rows[0])}")

traces = [json.loads(line) for line in trace_path.read_text(encoding="utf-8").splitlines() if line.strip()]
tkis = [item for item in traces if item.get("task") == "TKIS"]
if len(tkis) != 50:
    raise RuntimeError(f"query trace phải có 50 TKIS, nhận {len(tkis)}")
bad_bge = [item.get("query_id") for item in tkis if item.get("bge", {}).get("dense_status") != "passed" or item.get("bge", {}).get("reranker", {}).get("status") != "passed"]
if bad_bge:
    raise RuntimeError(f"BGE chưa chạy thành công cho TKIS: {bad_bge[:10]}")

print({
    "submission": str(SUBMISSION_PATH),
    "submission_queries": len(rows) - 1,
    "answers_per_query": len(rows[0]) - 1,
    "tkis_bge_verified": len(tkis),
    "bge_manifest": str(bge_manifest_path),
})


## 6. Smoke riêng từng task: KIS, AVS và QA

Cell này dùng artifacts vừa tạo. KIS/AVS kiểm tra parser → router → BGE retrieval/rerank → evidence. QA chạy cùng chuỗi rồi gọi Qwen3.5 4-bit và buộc trả grounded answer hoặc `insufficient_evidence`. Vì `QA_ANSWER_MODE=required`, lỗi load/inference sẽ làm cell fail nhưng vẫn ghi evidence vào `task_smoke.json`.


In [ ]:
if RUN_TASK_SMOKE:
    artifact_tag = "siglip2_so400m_patch16_384"
    smoke_env = os.environ.copy()
    smoke_env.update({
        "PYTHONUTF8": "1",
        "TOKENIZERS_PARALLELISM": "false",
        "HF_HOME": str(Path(MODEL_CACHE_ROOT) / "huggingface"),
        "RETRIEVAL_INDEX_PATH": str(RUN_ROOT / "indexes" / f"{artifact_tag}_flat_ip.faiss"),
        "RETRIEVAL_FRAME_MAP_PATH": str(RUN_ROOT / "metadata" / f"{artifact_tag}_frame_map.json"),
        "RETRIEVAL_MANIFEST_PATH": str(RUN_ROOT / "metadata" / f"{artifact_tag}_faiss_manifest.json"),
        "RETRIEVAL_TEXT_INDEX_PATH": str(RUN_ROOT / "indexes" / "retrieval_text_index.json"),
        "RETRIEVAL_MODEL_CACHE_DIR": str(Path(MODEL_CACHE_ROOT) / "huggingface"),
        "RETRIEVAL_DEVICE": DEVICE,
        "QA_TYPED_PARSER_ENABLED": "true",
        "QA_ROUTER_ENABLED": "true",
        "QA_EVIDENCE_BUNDLE_ENABLED": "true",
        "QA_BGE_DENSE_ENABLED": "true",
        "QA_BGE_RERANKER_ENABLED": "true",
        "QA_BGE_INDEX_ROOT": str(RUN_ROOT / "indexes" / "bge_m3"),
        "QA_BGE_MODEL_REVISION": BGE_M3_REVISION,
        "QA_BGE_RERANKER_MODEL": BGE_RERANKER_MODEL,
        "QA_BGE_RERANKER_REVISION": BGE_RERANKER_REVISION,
        "QA_BGE_RERANKER_ALPHA": str(BGE_RERANKER_ALPHA),
        "QA_BGE_BATCH_SIZE": str(BGE_BATCH_SIZE),
        "QA_BGE_DEVICE": DEVICE,
        "QA_BGE_MODEL_CACHE_DIR": str(Path(MODEL_CACHE_ROOT) / "bge_m3"),
        "QA_ANSWER_MODE": QA_ANSWER_MODE,
        "QA_ANSWER_MODEL": QA_MODEL,
        "QA_ANSWER_MODEL_REVISION": QA_MODEL_REVISION,
        "QA_ANSWER_DEVICE": DEVICE,
        "QA_ANSWER_QUANTIZATION": QA_QUANTIZATION,
        "QA_ANSWER_MODEL_CACHE_DIR": str(Path(MODEL_CACHE_ROOT) / "qa_answer"),
        "QA_ANSWER_CACHE_DIR": str(RUN_ROOT / "cache" / "qa_answers"),
        "QA_EXPERIMENT_ID": f"{RUN_ID}-task-smoke",
    })
    smoke_command = [
        sys.executable, "-m", "backend.app.services.retrieval.run_task_smoke",
        "--task", TASK_SMOKE,
        "--top-k", str(TASK_TOP_K),
        "--kis-query", KIS_QUERY,
        "--avs-query", AVS_QUERY,
        "--qa-query", QA_QUERY,
        "--output", str(TASK_SMOKE_PATH),
    ]
    subprocess.run(smoke_command, cwd=WORKSPACE, env=smoke_env, check=True)
    smoke = json.loads(TASK_SMOKE_PATH.read_text(encoding="utf-8"))
    print(json.dumps(smoke, ensure_ascii=False, indent=2)[:12000])
else:
    print("Task smoke disabled")


## Kết quả cần bàn giao

- `results/submission.csv`: schema và output public giữ nguyên.
- `results/query_traces.jsonl`: trace từng query, dùng để xác nhận BGE thực sự chạy.
- `indexes/bge_m3/`: FAISS dense text index + map + manifest.
- `results/task_smoke.json`: KIS/AVS evidence và grounded QA answer.
- `run_manifest.json`: trạng thái 10 stage để resume.

Notebook chỉ xác nhận execution/contract. Không có ground truth public nên không được dùng kết quả smoke để tuyên bố model mới tốt hơn.
